# <span style="color:DodgerBlue">**IV. Optimization of parameters**</span>

---

In the previous [tutorial](https://github.com/kenoz/NRT-tutorial/blob/colab_version/03_vi_time-series_plot.ipynb), we can see that using the default parameters in each of the methods is not really efficient. the detection dates can be very far from the reference date. It also happens that the methods are too sensitive to the slightest variations in the time-series. We therefore propose here to optimize the parameters to minimize the difference between the reference date and the detection date.

---

# <span style="color:DodgerBlue">0. Install NRT package (optional)</span>

Depending of your Python environment, you can install the NRT Package in different ways. Here, we choose to install the package in our current session (uncomment the line of code if necessary).

In [ ]:
!pip install nrt

---

## <span style="color:DodgerBlue">1. Import librairies</span>
First, we import some basic librairies.

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

# data storage via Google Drive
from google.colab import drive
#from IPython.display import display

We also import the spatial librairies

In [ ]:
import datetime as dt
import geopandas as gpd
import xarray as xr

...And an add-on to the NRT package, available only in this tutorial. It allows you to test different parameters on NRT methods in order to minimize the lag between the reference date and the detection date.

In [ ]:
!wget https://raw.githubusercontent.com/kenoz/NRT-tutorial/colab_version/nrt_utils.py
from nrt_utils import Nrt_run, params_bounds

---

## <span style="color:DodgerBlue">2. Loading samples and VI time-series</span>

### <span style="color:DodgerBlue">2.1. Point samples</span>

We are working here on a sample of 50 points. Each point is described by an identifier (`id`) and a date (`SAMPLE_1`) of dieback beginning obtained with visual interpretation.

In [ ]:
!mkdir -p nrt_data
![ ! -f nrt_data/samples_02.geojson ] && wget https://raw.githubusercontent.com/kenoz/NRT-tutorial/colab_version/data_ref/samples_02.geojson -P nrt_data

In [ ]:
# open samples

vector = gpd.read_file(r"nrt_data/samples_02.geojson")
vector.head()

For this exercise, we decide to keep 70% of the points for optimizing the method parameters and 20% for quality control.

In [ ]:
vector_train=vector.sample(frac=0.7)
vector_test=vector.drop(vector_train.index)

print('training samples:', len(vector_train))
print('test samples    :', len(vector_test))

The method `Nrt_run` of `nrt_utils` need two information:
- the field name of the sampling vector describing the reference date
- the "pivot" date that separates the fitting and the monitoring periods

In [ ]:
fdate = 'SAMPLE_1'
# pivot date (yyyy, mm, dd) fit/monitor
pdate = dt.datetime(2018, 6, 30)


### <span style="color:DodgerBlue">2.2. Loading VI time-series</span>

In this example, we test one of the avalaible datasets:

- vi-mask: VI time-series with cloud masks applied

In [ ]:
# mounting
mount_dir = '/content/drive'
drive.mount(mount_dir)

work_dir = os.path.join(mount_dir, 'My Drive/nrt_data')
output_dir = os.path.join(work_dir, 'output')

with open(os.path.join(output_dir, 'nrt_var.txt'), 'rb') as f:
    dict_var = pickle.load(f)

startdate = dict_var['startdate']
enddate = dict_var['enddate']
output_dir = dict_var['output_dir']

In [ ]:
cld_max = 0.5
vi_mask = xr.open_dataset(
    os.path.join(output_dir, f'S2TS_{startdate}-{enddate}_vi-cloud_inf_{cld_max}.nc')
    )
crswir = vi_mask.crswir

---

## <span style="color:DodgerBlue">3. Finding the most suitable parameters</span>
### <span style="color:DodgerBlue">3.1. Generating different parameters configurations</span>

Here, for each method's parameter, we specify the minimum value, the maximum value and the step. Then, we call the `params_bounds` function to generate all the possible parameters combinations.

In [ ]:
method_iqr = 'IQR'
params_config_iqr = {'sensitivity': (0.0, 5.0, 0.2),
                     'boundary': (1, 7, 2)}
params_test_iqr = params_bounds(params_config_iqr)
print(f"total number of {method_iqr} tests: {len(params_test_iqr)}")

method_ewma = 'EWMA'
params_config_ewma = {'sensitivity': (1, 10, 2),
                      'lambda_': (0, 1, 0.25),
                      'threshold_outlier': (1, 20, 2)}
params_test_ewma = params_bounds(params_config_ewma)
print(f"total number of {method_ewma} tests: {len(params_test_ewma)}")

method_cusum = 'CUSUM'
params_config_cusum = {'sensitivity': (0.0, 10.0, 0.2)}
params_test_cusum = params_bounds(params_config_cusum)
print(f"total number of {method_cusum} tests: {len(params_test_cusum)}")

method_mosum = 'MOSUM'
params_config_mosum = {'sensitivity': (0.001, 0.05, 0.001),
                       'h': [0.25, 0.5, 1]}
params_test_mosum = params_bounds(params_config_mosum)
print(f"total number of {method_mosum} tests: {len(params_test_mosum)}")

### <span style="color:DodgerBlue">3.2. Testing in parallel</span>

To improve the calculation, the tests are distributed with the `multipropcessing` Python package. So, we can define the number of processors to use.

In [ ]:
nb_process = 10

#### <span style="color:DodgerBlue">3.2.1. IQR method</span>

Now, the function `nrt_stat_in_parallel` runs the method with each parameters configuration over the training sampling points.

In [ ]:
if __name__ == "__main__":  # mandatory for using multiproc pooling
    optim_iqr = Nrt_run(crswir, method_iqr, pdate, vector_train, fdate)
    results_iqr = optim_iqr.nrt_stat_in_parallel(params_test_iqr, nb_process)

results_iqr.to_csv(f'{output_dir}/optim_{method_iqr}_crswir.csv')
results_iqr.head()

At the end, we deliver summary statistics for each parameters configuration.

In [ ]:
fields = ['mean', 'std', 'min', '25%', '50%', '75%', 'max']

for i in fields:
    print(f"parameters for min value in {i}:")
    display(results_iqr.loc[results_iqr[i] == results_iqr[i].min()].sort_values(by=['mean']).head())

Here, we can see that the configuration *"sensitivity: **1.2** | boundary: **5**"* seems to give the best results. On **average** the lag between the reference date and the detection date is about **45 / 50 days** and for 75% of points, it does not exceed 60 / 70 days.

Now we can compare these results with the test dataset:

In [ ]:
params_config_iqr = {'sensitivity': [1.2],
                     'boundary': [5]}
params_test_iqr = params_bounds(params_config_iqr)
optim_iqr = Nrt_run(crswir, method_iqr, pdate, vector_test, fdate)
results_iqr = optim_iqr.nrt_stat(params_test_iqr[0])

print(f"Statistics for the selected parameters:{params_test_iqr[0]}\n---")
results_iqr

#### <span style="color:DodgerBlue">3.2.2. EWMA method</span>

In [ ]:
if __name__ == "__main__":
    optim_ewma = Nrt_run(crswir, method_ewma, pdate, vector_train, fdate)
    results_ewma = optim_ewma.nrt_stat_in_parallel(params_test_ewma, nb_process)

results_ewma.to_csv(f'{output_dir}/optim_{method_ewma}_crswir.csv')
results_ewma.head()

In [ ]:
for i in fields:
    print(f"parameters for min value in {i}:")
    display(results_ewma.loc[results_ewma[i] == results_ewma[i].min()].sort_values(by=['mean']).head())

#### <span style="color:DodgerBlue">3.2.3. CUSUM method</span>

In [ ]:
if __name__ == "__main__":
    optim_cusum = Nrt_run(crswir, method_cusum, pdate, vector_train, fdate)
    results_cusum = optim_cusum.nrt_stat_in_parallel(params_test_cusum, nb_process)

results_cusum.to_csv(f'{output_dir}/optim_{method_cusum}_crswir.csv')
results_cusum.head()

In [ ]:
for i in fields:
    print(f"parameters for min value in {i}:")
    display(results_cusum.loc[results_cusum[i] == results_cusum[i].min()].sort_values(by=['mean']).head())


#### <span style="color:DodgerBlue">3.2.4. MOSUM method</span>

In [ ]:
if __name__ == "__main__":
    optim_mosum = Nrt_run(crswir, method_mosum, pdate, vector_train, fdate)
    results_mosum = optim_mosum.nrt_stat_in_parallel(params_test_mosum, nb_process)

results_mosum.to_csv(f'{output_dir}/optim_{method_mosum}_crswir.csv')
results_mosum.head()

In [ ]:
for i in fields:
    print(f"parameters for min value in {i}:")
    display(results_mosum.loc[results_mosum[i] == results_mosum[i].min()].sort_values(by=['mean']).head())